# Customer churn analysis

This notebook is a short walkthrough of the same workflow used by the command-line package. The model code lives in `src/` so the notebook stays reproducible rather than becoming a second implementation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from churn_prediction.data import generate_demo_data
from churn_prediction.modeling import train

## Load the data
The generated data includes a small amount of missing data so that the preprocessing steps are exercised.

In [ ]:
df = generate_demo_data(3000, random_state=42)
df.head()

In [ ]:
df.info()
df['churn'].value_counts(normalize=True).rename('share')

## Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='churn', ax=axes[0])
sns.boxplot(data=df, x='churn', y='tenure_months', ax=axes[1])
axes[0].set_title('Churn distribution')
axes[1].set_title('Tenure by churn status')
plt.tight_layout();

In [ ]:
contract_rate = (df.assign(churned=df.churn.eq('yes'))
                 .groupby('contract_type')['churned'].mean().sort_values())
contract_rate.plot.barh(title='Churn rate by contract', xlabel='Churn rate');

## Train and evaluate
The training function splits the data, compares candidate models with stratified cross-validation, tunes the winner, evaluates it on the held-out test set, and saves the outputs.

In [ ]:
report = train(df, ROOT / 'artifacts', cv_folds=5)
report

## Interpretation
ROC-AUC is used for model selection, but it is not enough on its own. Recall shows how many churners are identified, while precision shows how many contacted customers are actually likely to churn. In a real retention campaign, I would select the decision threshold using offer cost, expected customer value, and the number of customers the team can contact.